In [ ]:
import os
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

os.environ['OPENAI_API_KEY'] = ""

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
query_result = embeddings.embed_query("저는 배가 고파요")
print(type(query_result))

<class 'list'>


In [18]:
data = [
    '저는 배가 고파요',
    '저기 배가 지나가네요',
    '굶어서 허기가 지네요',
    '허기 워기라는 게임이 있는데 즐거워',
    '스팀에서 재밌는거 해야지',
    '스팀에어프라이로 연어구이 해먹을 거야'
]

df = pd.DataFrame(data, columns=['text'])
df

,text
0,저는 배가 고파요
1,저기 배가 지나가네요
2,굶어서 허기가 지네요
3,허기 워기라는 게임이 있는데 즐거워
4,스팀에서 재밌는거 해야지
5,스팀에어프라이로 연어구이 해먹을 거야


In [19]:
# 각 text 열에 존재하는 텍스트 데이터들을 get_embedding() 함수로 임벹딩해서 벡터로 변환하고, 이를 새로운 embedding 열을 만들어 저장함.
def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row : get_embedding(row.text), axis=1)
df

,text,embedding
0,저는 배가 고파요,"[-0.01663736067712307, -0.02178889885544777, 0..."
1,저기 배가 지나가네요,"[-0.003291434608399868, -0.02751476690173149, ..."
2,굶어서 허기가 지네요,"[-0.006181030999869108, -0.0069507937878370285..."
3,허기 워기라는 게임이 있는데 즐거워,"[-0.011329255998134613, -0.011715852655470371,..."
4,스팀에서 재밌는거 해야지,"[-0.015855437144637108, -0.011868358589708805,..."
5,스팀에어프라이로 연어구이 해먹을 거야,"[-0.006129896733909845, -0.02987600676715374, ..."


In [21]:
# cos_sim() 함수는 앞서 코사인 유사도를 계산하는 함수 cos_sim을 다시 한번 구현한 것임.
# return_answer_candidate() 함수는 임의의 검색어가 들어오면 해당 검색어를 get_embedding() 함수로 임베딩해서 벡터로 변환하고, query_embedding 변수에 저장함.
# 그 다음 현재 데이터 프레임 df 에 존재하는 모든 embedding 열의 벡터들과 코사인 유사도를 계산해 유사도가 가장 높은 3개의 데이터를 찾아 반환함.
def cos_sim(A, B):
    return dot(A, B)/ norm(A) * norm(B)

def return_answer_candidate(df, query):
    query_embedding = get_embedding(query)

    df["similarity"] = df['embedding'].apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc


sim_result = return_answer_candidate(df, '아무것도 안먹었더니 꼬르륵 소리가 나네')
sim_result


,text,embedding,similarity
2,굶어서 허기가 지네요,"[-0.006181030999869108, -0.0069507937878370285...",0.839535
5,스팀에어프라이로 연어구이 해먹을 거야,"[-0.006129896733909845, -0.02987600676715374, ...",0.827009
0,저는 배가 고파요,"[-0.01663736067712307, -0.02178889885544777, 0...",0.812785
